# 📊 Exploratory Data Analysis — Credit Card Fraud Detection

## Audit Risk Analytics Project

**Dataset:** Credit Card Fraud Detection (Kaggle / ULB)  
**Records:** ~284K European cardholder transactions (Sept 2013)  
**Fraud Rate:** ~0.17% (highly imbalanced)

This notebook explores transaction patterns, class distributions, temporal trends, and amount distributions to inform our anomaly detection and risk scoring models.

---

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import duckdb
import warnings
warnings.filterwarnings('ignore')

# Style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

print('✅ Libraries loaded')

## 1. Data Loading & First Look
Load the cleaned, transformed dataset from our ETL pipeline.

In [ ]:
from src.config import PROCESSED_PARQUET, TARGET_COL, PCA_FEATURES

df = pd.read_parquet(PROCESSED_PARQUET)
print(f'Shape: {df.shape}')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

## 2. Class Distribution (Fraud vs Legitimate)
The dataset is **highly imbalanced** — this is typical in real audit scenarios where fraudulent transactions are rare but critical to detect.

In [ ]:
fraud_counts = df[TARGET_COL].value_counts()
fraud_pct = df[TARGET_COL].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(['Legitimate (0)', 'Fraud (1)'], fraud_counts.values,
            color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_title('Transaction Count by Class', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
for i, (count, pct) in enumerate(zip(fraud_counts.values, fraud_pct.values)):
    label = f'{count:,}' + chr(10) + f'({pct:.3f}%)'
    axes[0].text(i, count + 1000, label, ha='center', fontsize=11)

# Pie chart
axes[1].pie(fraud_counts.values, labels=['Legitimate', 'Fraud'],
            autopct='%1.3f%%', colors=colors, explode=[0, 0.1],
            shadow=True, startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Class Distribution', fontsize=14, fontweight='bold')

plt.suptitle('Extreme Class Imbalance - Key Audit Consideration',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Fraud rate: {fraud_pct[1]:.4f}% - only {fraud_counts[1]} out of {len(df):,} transactions')
print(f'Imbalance ratio: 1:{int(fraud_counts[0]/fraud_counts[1])}')

## 3. Transaction Amount Analysis
Understanding the distribution of transaction amounts — critical for **materiality assessment** in audit.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Overall distribution
axes[0,0].hist(df['Amount'], bins=100, color='steelblue',
              edgecolor='black', linewidth=0.3, alpha=0.8)
axes[0,0].set_title('Transaction Amount Distribution', fontweight='bold')
axes[0,0].set_xlabel('Amount (EUR)')
axes[0,0].set_ylabel('Frequency')
mean_val = df['Amount'].mean()
med_val = df['Amount'].median()
axes[0,0].axvline(mean_val, color='red', linestyle='--',
                  label=f'Mean: EUR {mean_val:.2f}')
axes[0,0].axvline(med_val, color='orange', linestyle='--',
                  label=f'Median: EUR {med_val:.2f}')
axes[0,0].legend()

# Log-transformed
axes[0,1].hist(df['amount_log'], bins=100, color='#3498db',
              edgecolor='black', linewidth=0.3, alpha=0.8)
axes[0,1].set_title('Log-Transformed Amount Distribution', fontweight='bold')
axes[0,1].set_xlabel('log(1 + Amount)')
axes[0,1].set_ylabel('Frequency')

# By class
for cls, color, label in [(0, '#2ecc71', 'Legitimate'), (1, '#e74c3c', 'Fraud')]:
    subset = df[df[TARGET_COL] == cls]
    axes[1,0].hist(subset['Amount'], bins=80, alpha=0.6,
                  color=color, label=label, edgecolor='black', linewidth=0.3)
axes[1,0].set_title('Amount Distribution by Class', fontweight='bold')
axes[1,0].set_xlabel('Amount (EUR)')
axes[1,0].legend()
axes[1,0].set_xlim(0, 2000)

# Box plot by class
fraud_data = [df[df[TARGET_COL]==0]['Amount'].values,
              df[df[TARGET_COL]==1]['Amount'].values]
bp = axes[1,1].boxplot(fraud_data, labels=['Legitimate', 'Fraud'],
                       patch_artist=True, showfliers=False)
bp['boxes'][0].set_facecolor('#2ecc71')
bp['boxes'][1].set_facecolor('#e74c3c')
axes[1,1].set_title('Amount Box Plot by Class (outliers hidden)', fontweight='bold')
axes[1,1].set_ylabel('Amount (EUR)')

plt.suptitle('Amount Analysis - Materiality Assessment',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Amount statistics by class
amount_stats = df.groupby(TARGET_COL)['Amount'].agg(
    ['count', 'mean', 'median', 'std', 'min', 'max']).round(2)
amount_stats.index = ['Legitimate', 'Fraud']
print('Amount Statistics by Class:')
print(amount_stats.to_string())
fraud_mean = amount_stats.loc['Fraud', 'mean']
legit_mean = amount_stats.loc['Legitimate', 'mean']
print(f'\nFraud mean amount (EUR {fraud_mean:.2f}) vs Legitimate (EUR {legit_mean:.2f})')

### 3.1 Materiality Buckets
Grouping transactions by amount thresholds — a standard audit practice for risk assessment.

In [ ]:
bucket_stats = df.groupby('amount_bucket').agg(
    total_transactions=('Amount', 'count'),
    fraud_count=(TARGET_COL, 'sum'),
    avg_amount=('Amount', 'mean'),
).reset_index()
bucket_stats['fraud_rate_pct'] = (
    bucket_stats['fraud_count'] / bucket_stats['total_transactions'] * 100
).round(4)

bucket_order = ['low', 'medium', 'high', 'very_high']
bucket_stats['amount_bucket'] = pd.Categorical(
    bucket_stats['amount_bucket'], categories=bucket_order, ordered=True)
bucket_stats = bucket_stats.sort_values('amount_bucket')

print('Materiality Bucket Analysis:')
print(bucket_stats.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#27ae60', '#f39c12', '#e67e22', '#e74c3c']

axes[0].bar(bucket_stats['amount_bucket'].astype(str),
            bucket_stats['total_transactions'], color=colors,
            edgecolor='black', linewidth=0.5)
axes[0].set_title('Transaction Count by Materiality Bucket', fontweight='bold')
axes[0].set_ylabel('Count')

axes[1].bar(bucket_stats['amount_bucket'].astype(str),
            bucket_stats['fraud_rate_pct'], color=colors,
            edgecolor='black', linewidth=0.5)
axes[1].set_title('Fraud Rate (%) by Materiality Bucket', fontweight='bold')
axes[1].set_ylabel('Fraud Rate (%)')

plt.suptitle('Materiality Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 4. Temporal Analysis
Understanding when transactions and fraud occur — critical for identifying **off-hours risk patterns**.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Transactions by hour
hourly = df.groupby('time_hour_int').agg(
    total=('Amount', 'count'),
    fraud=(TARGET_COL, 'sum')
).reset_index()
hourly['fraud_rate'] = hourly['fraud'] / hourly['total'] * 100

axes[0,0].bar(hourly['time_hour_int'], hourly['total'],
             color='steelblue', edgecolor='black', linewidth=0.3)
axes[0,0].axvspan(0, 8, alpha=0.15, color='red', label='Off-hours')
axes[0,0].axvspan(18, 24, alpha=0.15, color='red')
axes[0,0].set_title('Transaction Volume by Hour', fontweight='bold')
axes[0,0].set_xlabel('Hour of Day')
axes[0,0].set_ylabel('Transaction Count')
axes[0,0].legend()

# Fraud by hour
axes[0,1].bar(hourly['time_hour_int'], hourly['fraud'],
             color='#e74c3c', edgecolor='black', linewidth=0.3)
axes[0,1].axvspan(0, 8, alpha=0.15, color='red')
axes[0,1].axvspan(18, 24, alpha=0.15, color='red')
axes[0,1].set_title('Fraud Count by Hour', fontweight='bold')
axes[0,1].set_xlabel('Hour of Day')
axes[0,1].set_ylabel('Fraud Count')

# Fraud rate by hour
axes[1,0].plot(hourly['time_hour_int'], hourly['fraud_rate'],
             'o-', color='#e74c3c', linewidth=2, markersize=6)
axes[1,0].fill_between(hourly['time_hour_int'], hourly['fraud_rate'],
                      alpha=0.2, color='#e74c3c')
axes[1,0].axvspan(0, 8, alpha=0.1, color='red', label='Off-hours')
axes[1,0].axvspan(18, 24, alpha=0.1, color='red')
axes[1,0].set_title('Fraud Rate (%) by Hour', fontweight='bold')
axes[1,0].set_xlabel('Hour of Day')
axes[1,0].set_ylabel('Fraud Rate (%)')
axes[1,0].legend()

# Business hours vs off-hours
bh_stats = df.groupby('is_business_hours').agg(
    total=('Amount', 'count'),
    fraud=(TARGET_COL, 'sum')
).reset_index()
bh_stats['fraud_rate'] = bh_stats['fraud'] / bh_stats['total'] * 100
labels = ['Off-Hours', 'Business Hours']
axes[1,1].bar(labels, bh_stats['fraud_rate'],
             color=['#e74c3c', '#2ecc71'], edgecolor='black', linewidth=0.5)
axes[1,1].set_title('Fraud Rate: Business Hours vs Off-Hours', fontweight='bold')
axes[1,1].set_ylabel('Fraud Rate (%)')
for i, val in enumerate(bh_stats['fraud_rate']):
    axes[1,1].text(i, val + 0.01, f'{val:.3f}%', ha='center',
                  fontsize=12, fontweight='bold')

plt.suptitle('Temporal Risk Analysis', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
segment_stats = df.groupby('time_segment', observed=True).agg(
    transactions=('Amount', 'count'),
    fraud_count=(TARGET_COL, 'sum'),
    avg_amount=('Amount', 'mean')
).reset_index()
segment_stats['fraud_rate_pct'] = (
    segment_stats['fraud_count'] / segment_stats['transactions'] * 100
).round(4)
print('Time Segment Analysis:')
print(segment_stats.to_string(index=False))

## 5. Correlation Analysis
Examining which features are most correlated with fraud — guides feature selection for our models.

In [ ]:
feature_cols = PCA_FEATURES + ['Amount', 'amount_log', 'amount_zscore',
                               'time_hour', 'pca_magnitude']
correlations = df[feature_cols + [TARGET_COL]].corr()[TARGET_COL].drop(
    TARGET_COL).sort_values()

fig, ax = plt.subplots(figsize=(12, 8))
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in correlations.values]
ax.barh(correlations.index, correlations.values, color=colors,
        edgecolor='black', linewidth=0.3)
ax.set_title('Feature Correlation with Fraud (Class)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Pearson Correlation')
ax.axvline(x=0, color='black', linewidth=0.8)

plt.tight_layout()
plt.show()

print('Top 5 positive correlations:')
print(correlations.tail(5).to_string())
print('\nTop 5 negative correlations:')
print(correlations.head(5).to_string())

## 6. PCA Feature Distributions (Fraud vs Legitimate)
Comparing distributions of the most discriminative PCA features between fraud and legitimate transactions.

In [ ]:
top_corr = correlations[PCA_FEATURES].abs().sort_values(
    ascending=False).head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for idx, feat in enumerate(top_corr):
    ax = axes[idx // 3][idx % 3]
    for cls, color, label in [(0, '#2ecc71', 'Legit'),
                              (1, '#e74c3c', 'Fraud')]:
        subset = df[df[TARGET_COL] == cls][feat]
        ax.hist(subset, bins=50, alpha=0.6, color=color,
               label=label, density=True, edgecolor='black', linewidth=0.2)
    ax.set_title(f'{feat} Distribution', fontweight='bold')
    ax.legend()
    corr_val = correlations[feat]
    ax.text(0.02, 0.95, f'r = {corr_val:.3f}', transform=ax.transAxes,
           fontsize=10, verticalalignment='top',
           bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Top Discriminative PCA Features',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 7. Amount Outlier Analysis
Identifying transactions that deviate significantly from the norm — a core audit analytics technique.

In [ ]:
outlier_df = df[df['is_outlier_amount']].copy()
non_outlier_df = df[~df['is_outlier_amount']].copy()

print(f'Amount outliers (|z| > 3): {len(outlier_df):,} '
      f'({len(outlier_df)/len(df)*100:.2f}%)')
print(f'Fraud among outliers: {outlier_df[TARGET_COL].sum()} '
      f'({outlier_df[TARGET_COL].mean()*100:.2f}%)')
print(f'Fraud among non-outliers: {non_outlier_df[TARGET_COL].sum()} '
      f'({non_outlier_df[TARGET_COL].mean()*100:.2f}%)')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: Amount vs PCA magnitude
scatter = axes[0].scatter(df['Amount'], df['pca_magnitude'],
                         c=df[TARGET_COL], cmap='RdYlGn_r',
                         alpha=0.3, s=2, edgecolors='none')
axes[0].set_title('Amount vs PCA Magnitude', fontweight='bold')
axes[0].set_xlabel('Amount (EUR)')
axes[0].set_ylabel('PCA Magnitude')
axes[0].set_xlim(0, 3000)
plt.colorbar(scatter, ax=axes[0], label='Fraud')

# Z-score distribution
axes[1].hist(df['amount_zscore'], bins=100, color='steelblue',
            edgecolor='black', linewidth=0.2, alpha=0.8)
axes[1].axvline(3, color='red', linestyle='--', linewidth=2,
               label='z = 3 (outlier threshold)')
axes[1].axvline(-3, color='red', linestyle='--', linewidth=2)
axes[1].set_title('Amount Z-Score Distribution', fontweight='bold')
axes[1].set_xlabel('Z-Score')
axes[1].set_ylabel('Frequency')
axes[1].set_xlim(-5, 10)
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. DuckDB Analytical Queries
Running SQL-style analytics on the dataset — demonstrating the kind of queries an auditor would use.

In [ ]:
con = duckdb.connect()
con.register('transactions', df)

# Query 1: High-risk summary
print('=== High-Risk Transaction Summary ===')
result = con.execute("""
    SELECT 
        amount_bucket,
        COUNT(*) as total_txn,
        SUM(Class) as fraud_count,
        ROUND(AVG(Amount), 2) as avg_amount,
        ROUND(SUM(Class) * 100.0 / COUNT(*), 4) as fraud_rate_pct
    FROM transactions
    GROUP BY amount_bucket
    ORDER BY fraud_rate_pct DESC
""").fetchdf()
print(result.to_string(index=False))

In [ ]:
# Query 2: Hourly risk profile
print('=== Hourly Risk Profile ===')
result = con.execute("""
    SELECT 
        time_hour_int as hour,
        COUNT(*) as transactions,
        SUM(Class) as fraud_cases,
        ROUND(AVG(Amount), 2) as avg_amount,
        ROUND(SUM(Class) * 100.0 / COUNT(*), 4) as fraud_rate_pct,
        CASE 
            WHEN time_hour_int BETWEEN 8 AND 17 THEN 'Business'
            ELSE 'Off-Hours'
        END as period
    FROM transactions
    GROUP BY time_hour_int
    ORDER BY fraud_rate_pct DESC
    LIMIT 10
""").fetchdf()
print(result.to_string(index=False))

In [ ]:
# Query 3: Anomaly candidates
print('=== Top 20 Anomalous Transactions ===')
result = con.execute("""
    SELECT 
        Amount,
        ROUND(amount_zscore, 2) as z_score,
        amount_bucket,
        time_hour_int as hour,
        is_business_hours,
        Class as is_fraud,
        ROUND(pca_magnitude, 2) as pca_mag
    FROM transactions
    WHERE ABS(amount_zscore) > 3
    ORDER BY amount_zscore DESC
    LIMIT 20
""").fetchdf()
print(result.to_string(index=False))

## 9. Key Findings & Audit Implications

### Data Quality
- 283,726 clean transactions after removing 1,081 duplicates
- No missing values — excellent data quality
- Extreme class imbalance: only 0.17% fraud (473 cases)

### Amount Patterns
- Mean transaction ~EUR 88, highly right-skewed
- Fraud transactions show different amount distributions
- Very high-value transactions warrant closer audit scrutiny

### Temporal Patterns
- Transaction volume varies significantly by hour
- Off-hours may show different fraud rates — key risk indicator
- Night-time (00:00-06:00) patterns critical for control testing

### Feature Importance
- Several PCA components show strong fraud discrimination
- Amount z-score and PCA magnitude are useful anomaly indicators

### Next Steps
1. **Anomaly Detection** — Train Isolation Forest and LOF models
2. **Risk Scoring** — Build multi-factor risk engine
3. **Dashboard** — Visualise findings in Streamlit and Power BI